# Cross-Entropy (concept 37)

$H(p, q) = -\sum_x p(x) \log q(x)$ — the average code-length when using $q$ to encode samples drawn from $p$.

Gibbs' inequality: $H(p, q) \ge H(p)$, with equality iff $p = q$.

Decomposition: $H(p, q) = H(p) + D_{\mathrm{KL}}(p \| q)$.

This notebook (stdlib only) walks through the definitions, verifies Gibbs numerically, and shows the softmax-classification special case.

In [ ]:
from math import log2

def entropy(p):
    return -sum(pi * log2(pi) for pi in p if pi > 0.0)

def cross_entropy(p, q):
    total = 0.0
    for pi, qi in zip(p, q):
        if pi == 0.0:
            continue
        if qi == 0.0:
            return float('inf')
        total -= pi * log2(qi)
    return total

def kl_divergence(p, q):
    total = 0.0
    for pi, qi in zip(p, q):
        if pi == 0.0:
            continue
        if qi == 0.0:
            return float('inf')
        total += pi * log2(pi / qi)
    return total

## 1. Worked example: fair coin, biased code

$p = (0.5, 0.5)$, $q = (0.9, 0.1)$. Expect $H(p) = 1$ bit and $H(p, q) \approx 1.737$ bits.

In [ ]:
p = [0.5, 0.5]
q = [0.9, 0.1]
print(f'H(p)    = {entropy(p):.4f} bits')
print(f'H(p, q) = {cross_entropy(p, q):.4f} bits')
print(f'KL(p||q) = {kl_divergence(p, q):.4f} bits')
print(f'H(p, q) - H(p) - KL = {cross_entropy(p, q) - entropy(p) - kl_divergence(p, q):.2e}')

## 2. Verify Gibbs' inequality across a battery of pairs

For each $(p, q)$, check $H(p, q) \ge H(p)$ and $H(p, q) - H(p) = D_{\mathrm{KL}}(p \| q)$.

In [ ]:
pairs = [
    ([0.5, 0.5], [0.5, 0.5]),
    ([0.5, 0.5], [0.9, 0.1]),
    ([0.7, 0.3], [0.3, 0.7]),
    ([0.25]*4, [0.1, 0.2, 0.3, 0.4]),
]
for p, q in pairs:
    Hp = entropy(p); Hpq = cross_entropy(p, q); Dkl = kl_divergence(p, q)
    assert Hpq + 1e-12 >= Hp, 'Gibbs violated!'
    assert abs((Hpq - Hp) - Dkl) < 1e-9, 'Decomposition broken!'
    print(f'p={p}, q={q}: H(p)={Hp:.3f}, H(p,q)={Hpq:.3f}, KL={Dkl:.3f}')
print('All checks passed.')

## 3. Softmax classification: cross-entropy as the standard ML loss

Ground truth one-hot $p = [1, 0, 0, 0, 0]$, predicted softmax $q = [0.6, 0.1, 0.1, 0.1, 0.1]$.
Cross-entropy collapses to $-\log_2 q(y^*) = -\log_2 0.6 \approx 0.737$ bits.

In [ ]:
truth = [1.0, 0.0, 0.0, 0.0, 0.0]
pred  = [0.6, 0.1, 0.1, 0.1, 0.1]
ce = cross_entropy(truth, pred)
print(f'CE loss = {ce:.4f} bits  (== -log2(0.6) = {-log2(0.6):.4f})')

# Confidently wrong vs confidently right:
for q_true in [0.99, 0.6, 0.2, 0.01]:
    q_other = (1 - q_true) / 4
    pred = [q_true] + [q_other]*4
    print(f'q(true)={q_true:.2f}: CE={cross_entropy(truth, pred):.3f} bits')

## 4. Take-aways

- **Gibbs' inequality** $H(p, q) \ge H(p)$ makes cross-entropy a *valid* loss: minimising it cannot drive the loss below the irreducible $H(p)$.
- The **decomposition** $H(p, q) = H(p) + D_{\mathrm{KL}}(p \| q)$ shows that minimising CE over $q$ (with $p$ fixed) is the same as minimising KL.
- For one-hot labels, CE collapses to $-\log q(y^*)$, the negative log-likelihood — this is the loss used in classification, language modelling (per-token), and the final-step CE supervision in ELF (arxiv:2605.10938).
- Advantage-weighted CE (ReasonMaxxer §3.1) just multiplies each term by a sample weight, recovering policy-gradient updates.